# One GPU, a house model and many program specialists

## The story

A small shop serves **one house model** to all its chemists, and **one specialist per program**, all
from **one L4 GPU**. This uses Ray Serve **model multiplexing**: one deployment holds several OpenFold3
checkpoints by id and runs the one a request asks for.

- A **request names a model** by its program (routed to that program's current version) or by an exact
  version id.
- The **house model is just one id** in the pool: the default route.
- **Adding a new specialist version is a registry entry, not a swap.** Old versions stay reachable by
  id, so you can canary, roll back, and reproduce an old pose for an audit.
- Every response carries the **exact version** that answered.

Why this is the right shape for a small shop: each program alone can't fill a GPU, but pooled together
they keep one L4 busy. You pay for **one** always-on GPU, not one per program.

## This demo

One compound: EV-A71 2A **x7259a** (pKD 7.9). Boltz-2 adds the affinity. The pool holds:

| id | role | checkpoint | diffusion samples |
|---|---|---|---|
| `house` | general model, the default route | `of3-p2-145k.pt` | 1 |
| `ev71-v1` | the program's first specialist | our Ray Train fine-tune (or the public one until `OURS` is set) | 1 |
| `ev71-v2` | a newer version of it | the public fine-tune, more inference compute | 5 |

The house model gets x7259a wrong (~5 Å); both specialist versions get it right (~0.4 Å). The point of
this notebook is the **version lifecycle**: add a version, canary it, promote it, roll back, and keep
the old one reachable.

**Honest note:** OpenFold3 runs as a subprocess that loads the checkpoint from disk each call, so this
shows the multiplexing **routing and API**, not a warm-model speed-up. A real warm pool needs a
long-lived in-process model (a bigger change).

In [ ]:
import json, os, time
from pathlib import Path

import anyscale
import pandas as pd
import requests
from anyscale.service.models import ServiceConfig

SHARED = Path(os.environ.get("AIDDD_SHARED", "/mnt/shared_storage/aiddd"))
CKPTS = SHARED / "of3_cache"
assert (SHARED / "results" / "ev71-2a").exists(), f"missing {SHARED}/results: copy the data first"
assert (CKPTS / "of3-p2-145k.pt").exists(), "missing of3-p2-145k.pt: run scripts/setup_data.py"

# Your own Service. Separate from aiddd-viewer (the shared one) and aiddd-viewer-geoff (hot swap).
SERVICE_NAME = "aiddd-viewer-multi"

# x7259a: the house model is wrong, the specialist is right. Backups: x7570a, x7056a.
SMILES = "CSc1ncnc(N2CCC[C@@H]2C2CC2)c1CC(=O)N1CCC[C@H](C(N)=O)C1"

# Specialist v1: our Ray Train fine-tune of of3-p2-145k.pt. None falls back to the public one.
OURS = None   # e.g. SHARED / "finetune/runs/<run>/checkpoint_<n>/of3_ev71_ft.pt"
assert OURS is None or Path(OURS).exists(), f"missing {OURS}"
SPECIALIST = CKPTS / "of3p2-ft-ev2a.ckpt"

# The model registry: id -> weights and inference settings. A version is (checkpoint + samples),
# so "more diffusion samples" is a real, different version. With OURS set, v1 is our own fine-tune.
HOUSE = {"name": "House model: OF3-p2 145k", "checkpoint": str(CKPTS / "of3-p2-145k.pt"),
         "legacy": True, "samples": 1}
V1 = {"name": "EV-A71 specialist v1", "checkpoint": str(OURS or SPECIALIST), "legacy": True, "samples": 1}
V2 = {"name": "EV-A71 specialist v2 (more diffusion)", "checkpoint": str(SPECIALIST), "legacy": True, "samples": 5}

# Mutable state the cells below edit, then apply() in place.
REGISTRY = {"house": HOUSE}
ROUTES = {"default": "house"}
print("start:", list(REGISTRY), ROUTES)

## Helpers

`apply()` pushes the current `REGISTRY` and `ROUTES` to the running service, in place. `predict()`
calls `/predict`, picking a model by `program` or exact `model` id. `show_models()` reads `/models`:
the registry, the routes, and the pool's load log (one entry per real load; a reload after an eviction
appears twice).

In [ ]:
def service_config():
    """service_multi.yaml, renamed, with the current registry and routes for both deployments."""
    cfg = ServiceConfig.from_yaml("service_multi.yaml", name=SERVICE_NAME)
    deployments = [{"name": "ViewerMulti", "user_config": {"routes": ROUTES, "models": REGISTRY}},
                   {"name": "OF3Pool", "user_config": {"models": REGISTRY}}]
    return cfg.options(applications=[{**app, "deployments": deployments} for app in cfg.applications])


def apply():
    """Push REGISTRY and ROUTES to the service in place (no restart), and wait for RUNNING."""
    try:
        running = str(anyscale.service.status(name=SERVICE_NAME).state) == "RUNNING"
    except Exception:
        running = False
    anyscale.service.deploy(service_config(), in_place=running, name=SERVICE_NAME)
    anyscale.service.wait(name=SERVICE_NAME, state="RUNNING", timeout_s=1800 if not running else 600)


def _hdr():
    s = anyscale.service.status(name=SERVICE_NAME)
    return s.query_url.rstrip("/"), ({"Authorization": f"Bearer {s.query_auth_token}"} if s.query_auth_token else {})


def predict(smiles=SMILES, program=None, model=None):
    url, headers = _hdr()
    body = {"smiles": smiles, **({"program": program} if program else {}), **({"model": model} if model else {})}
    return requests.post(url + "/predict", json=body, headers=headers, timeout=600).json()


def predict_until(expect, **sel):
    """Predict until the pose comes from version `expect` (an in-place update can lag a few seconds)."""
    for _ in range(30):
        out = predict(**sel)
        if (out.get("models") or {}).get("pose", {}).get("version") == expect:
            return out
        time.sleep(4)
    raise TimeoutError(f"pose still not from {expect}: got {out.get('models')}")


def summary(out):
    key, m = out.get("answer_key") or {}, out.get("metrics", {})
    return {"model_id": out.get("model_id"), "version": (out.get("models") or {}).get("pose", {}).get("version"),
            "replica": out.get("replica"), "crystal": key.get("complex"),
            "ligand RMSD (A)": key.get("lig_rmsd"), "pose ipTM": out.get("pose_iptm"),
            "affinity (log10 IC50, uM)": m.get("affinity_pred_value"),
            "status": out.get("status"), "seconds": out.get("seconds")}


def show_models():
    url, headers = _hdr()
    d = requests.get(url + "/models", headers=headers, timeout=60).json()
    print("routes:", d["routes"])
    print("registry:", list(d["registry"]))
    print("replica:", d.get("replica"))
    print("load log:", [e["model"] for e in d.get("load_log", [])])
    return d

## 1. Start the Service: house model only

The first deploy starts one L4 node, about 5 to 10 minutes. If the Service is already running, this
cell only resets it to house-only, in place.

In [ ]:
REGISTRY = {"house": HOUSE}
ROUTES = {"default": "house"}
apply()
print(anyscale.service.status(name=SERVICE_NAME).query_url)

## 2. The house model on the new program's compound

Every request routes to `house`. It has never seen this pocket with a ligand in it.

In [ ]:
summary(predict_until("house"))

## 3. A new program: add its specialist and route to it

Add `ev71-v1` to the registry and point the `ev71-2a` program at it. No swap: `house` is still the
default for everyone else. This is one in-place update.

In [ ]:
REGISTRY["ev71-v1"] = V1
ROUTES["ev71-2a"] = "ev71-v1"
apply()
summary(predict_until("ev71-v1", program="ev71-2a"))

## 4. Multiplexing: house and specialist share one GPU

Send a default (house) request and an `ev71-2a` request. Both models are now loaded on the one pool
replica. The load log shows each one loading once.

In [ ]:
h = predict(program="default")
s = predict(program="ev71-2a")
display(pd.DataFrame({"house request": summary(h), "ev71-2a request": summary(s)}))
show_models()

## 5. Ship the next version, with no swap. Canary it.

Add `ev71-v2` to the registry. It does not replace v1: both are live. Before promoting it, query each
version **by exact id** and compare on the same molecule.

In [ ]:
REGISTRY["ev71-v2"] = V2
apply()
v1 = predict_until("ev71-v1", model="ev71-v1")
v2 = predict_until("ev71-v2", model="ev71-v2")
pd.DataFrame({"ev71-v1": summary(v1), "ev71-v2": summary(v2)})

## 6. Promote, roll back, audit

Promotion is a route change, not a swap. Roll back the same way. The old version stays reachable by its
id, so an old pose can always be reproduced.

In [ ]:
ROUTES["ev71-2a"] = "ev71-v2"   # promote
apply()
promoted = summary(predict_until("ev71-v2", program="ev71-2a"))

ROUTES["ev71-2a"] = "ev71-v1"   # roll back
apply()
rolled_back = summary(predict_until("ev71-v1", program="ev71-2a"))

ROUTES["ev71-2a"] = "ev71-v2"   # forward again
apply()
audit_old = summary(predict_until("ev71-v1", model="ev71-v1"))   # old version still reachable by id
pd.DataFrame({"after promote (route)": promoted, "after rollback (route)": rolled_back,
              "old version by id (audit)": audit_old})

## 7. The pool right now

The registry holds every version; the routes say which one each program gets; the load log shows what
this replica has loaded.

In [ ]:
show_models()

## 8. Reset and stop

Reset to house-only. Terminate when the session ends.

In [ ]:
REGISTRY = {"house": HOUSE}
ROUTES = {"default": "house"}
apply()

In [ ]:
# anyscale.service.terminate(name=SERVICE_NAME)